# Module 6.3 — Type Hints & Static Checking

### Python Mastery · Track 6: Testing, Tooling & DevOps

Type hints annotate your code with the types it expects. Python does not enforce them at runtime, but a **static type checker** such as `mypy` reads them and catches whole classes of bugs before the code ever runs. This module covers writing hints, the modern syntax, advanced types, and running `mypy`.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Annotated code runs normally in cells (hints are not enforced at runtime). To see type checking, the examples write a `.py` file and run `!python -m mypy` on it, so you observe real errors being caught.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Annotate variables, functions, and collections with types.
2. Use the modern syntax: built-in generics and `X | Y` unions.
3. Use `Optional`, `Any`, `Callable`, and type aliases.
4. Define structural types with `Protocol` and typed dicts with `TypedDict`.
5. Run `mypy` and interpret the errors it reports.

**Prerequisites:** Tracks 1 to 5.

---

## Part 1 · Annotating Functions and Variables

A type hint states the expected type of a variable, parameter, or return value, written after a colon, with `->` for return types. Hints are documentation that tools can check; they do **not** change how the code runs. The cell below uses hints and runs normally, because Python ignores them at runtime.

In [ ]:
def greet(name: str, times: int = 1) -> str:
    return (name + " ") * times

# A variable annotation (optional, but useful for clarity).
count: int = 3
message: str = greet("hello", count)
print(message)

# Hints are not enforced at runtime. The annotations below say int, but Python
# will happily concatenate two strings, since '+' is valid for strings too.
def add(a: int, b: int) -> int:
    return a + b

print(add("x", "y"))     # returns "xy": the wrong types are not blocked at runtime

The value of hints comes from a checker reading them. The cell below writes a file with a clear type error, then runs `mypy` on it, which reports the problem without the code being executed.

In [ ]:
%%writefile typed_basic.py
def add(a: int, b: int) -> int:
    return a + b

result: int = add(2, 3)       # fine
bad = add("x", "y")           # mypy will flag: str given where int expected

In [ ]:
# mypy analyses the file statically and reports the type error.
!python -m mypy typed_basic.py

## Part 2 · Collections and the Modern Syntax

Container types are annotated with the element types they hold. Since Python 3.9 you can use the built-in names directly (`list[int]`, `dict[str, float]`) rather than importing from `typing`. Since Python 3.10, a union of types is written `X | Y` instead of `Union[X, Y]`, which is shorter and clearer.

In [ ]:
# Built-in generics (Python 3.9+): no imports needed.
def total(values: list[float]) -> float:
    return sum(values)

def lookup(table: dict[str, int], key: str) -> int:
    return table.get(key, 0)

print(total([1.5, 2.5, 3.0]))
print(lookup({"a": 1, "b": 2}, "a"))

# The modern union syntax (Python 3.10+): a value that may be one of several types.
def parse(value: int | str) -> str:
    return str(value)

print(parse(42), parse("hi"))

## Part 3 · `Optional`, `Any`, `Callable`, and Aliases

Several constructs from `typing` (and modern equivalents) cover common needs:

- `Optional[T]`, equivalently `T | None`, means a value may be of type `T` or `None`.
- `Any` opts out of checking for a value; use sparingly, as it disables the checker there.
- `Callable[[args], return]` describes a function value.
- A **type alias** gives a meaningful name to a complex type.

In [ ]:
from typing import Optional, Any, Callable

# Optional: may be a str or None. (str | None means the same thing.)
def find_user(uid: int) -> Optional[str]:
    users = {1: "Asha", 2: "Ben"}
    return users.get(uid)            # returns None if not found

print(find_user(1), find_user(99))

# Callable: a parameter that is itself a function.
def apply(func: Callable[[int], int], value: int) -> int:
    return func(value)

print(apply(lambda x: x * 2, 10))

# A type alias names a complex type for readability.
Matrix = list[list[float]]
def first_row(m: Matrix) -> list[float]:
    return m[0]

print(first_row([[1.0, 2.0], [3.0, 4.0]]))

# Any disables checking for that value; use it only when truly necessary.
def passthrough(x: Any) -> Any:
    return x
print(passthrough("anything"))

## Part 4 · `Protocol` and `TypedDict`

Two powerful tools describe shapes rather than concrete classes:

- A `Protocol` defines a **structural** type: any object with the right methods or attributes matches it, regardless of its class. This formalises duck typing for the checker.
- A `TypedDict` describes a dictionary with **specific keys and value types**, useful for JSON-like records where a full class would be heavy.

In [ ]:
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> str:           # any object with a matching draw() satisfies this
        ...

def render(item: Drawable) -> str:
    return item.draw()

# These classes do not inherit from Drawable, yet they match it structurally.
class Circle:
    def draw(self) -> str:
        return "drawing a circle"

class Square:
    def draw(self) -> str:
        return "drawing a square"

print(render(Circle()))
print(render(Square()))

In [ ]:
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int
    active: bool

# A plain dict, but the checker knows exactly which keys and types are expected.
def describe(user: User) -> str:
    return f"{user['name']} ({user['age']}), active={user['active']}"

asha: User = {"name": "Asha", "age": 30, "active": True}
print(describe(asha))

## Part 5 · Running `mypy` and Reading Its Output

`mypy` checks a file or package and reports each type inconsistency with a file, line, and message. The example below packs several common mistakes into one file so you can see the kinds of errors it catches: wrong argument types, wrong return types, and calling a missing method.

In [ ]:
%%writefile typed_errors.py
def double(n: int) -> int:
    return n * 2

# Error 1: wrong argument type.
double("not a number")

# Error 2: wrong return type.
def get_name() -> str:
    return 42

# Error 3: assigning the wrong type.
age: int = "thirty"

# Error 4: a possibly-None value used without checking.
def find(uid: int) -> str | None:
    return {1: "Asha"}.get(uid)

length: int = len(find(1))    # find may return None; len(None) would fail

In [ ]:
# mypy reports each problem with its location. A clean codebase aims for zero errors.
!python -m mypy typed_errors.py

In a real project, `mypy` is run in continuous integration (Module 6.6) so that type errors block a merge, just as failing tests do. Teams often adopt types gradually, annotating new and critical code first, and tightening `mypy` settings over time.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Annotating a function (Easy)

In [ ]:
def repeat(text: str, times: int) -> str:
    return text * times

print(repeat("ab", 3))
# The hints document the contract; the function runs as normal.

### Example 2 — Typed collections (Easy)

In [ ]:
def average(scores: list[float]) -> float:
    return sum(scores) / len(scores)

print(average([80.0, 90.0, 100.0]))

### Example 3 — Optional and a guarded use (Medium)

In [ ]:
def first_even(numbers: list[int]) -> int | None:
    for n in numbers:
        if n % 2 == 0:
            return n
    return None

result = first_even([1, 3, 5, 8])
# Guard against None before using the value (mypy would require this).
if result is not None:
    print("first even:", result)
else:
    print("no even number found")

### Example 4 — A Protocol for structural typing (Medium)

In [ ]:
from typing import Protocol

class Sized(Protocol):
    def __len__(self) -> int:
        ...

def describe_size(item: Sized) -> str:
    return f"has {len(item)} elements"

# Lists, strings, and dicts all satisfy Sized structurally.
print(describe_size([1, 2, 3]))
print(describe_size("hello"))
print(describe_size({"a": 1}))

### Example 5 — A TypedDict with mypy verification (Difficult)

In [ ]:
%%writefile typed_record.py
from typing import TypedDict

class Product(TypedDict):
    name: str
    price: float
    in_stock: bool

def label(p: Product) -> str:
    status = "available" if p["in_stock"] else "sold out"
    return f"{p['name']}: {p['price']:.2f} ({status})"

ok: Product = {"name": "Book", "price": 29.99, "in_stock": True}
print(label(ok))

# This has a wrong value type and a missing key; mypy will flag both.
bad: Product = {"name": "Pen", "price": "cheap"}

In [ ]:
!python -m mypy typed_record.py

### Example 6 — Generics with a type variable (Difficult)

In [ ]:
%%writefile typed_generic.py
from typing import TypeVar

T = TypeVar("T")

def first(items: list[T]) -> T:
    """Return the first item, preserving its type."""
    return items[0]

# mypy infers the specific type for each call.
n: int = first([1, 2, 3])          # T is int here
s: str = first(["a", "b"])         # T is str here

# This line is a type error: assigning a str result to an int variable.
wrong: int = first(["x", "y"])

In [ ]:
!python -m mypy typed_generic.py

---

## Exercises

For checking, write a `.py` file with `%%writefile` and run `!python -m mypy` on it. For plain annotation practice, a normal cell is fine.

**Exercise 1 (Easy).** Annotate a function `multiply(a, b)` that takes two `int`s and returns an `int`. Call it and print the result.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Annotate a function `names_over(people, threshold)` that takes a `list[str]` and an `int` and returns a `list[str]` (you decide the logic). Demonstrate it.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a function `safe_divide(a, b)` annotated to return `float | None`, returning `None` when `b` is zero. Then use it with a guard that checks for `None` before printing.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Define a `Protocol` named `Closeable` requiring a `close() -> None` method, and a function `shut(item)` that calls it. Provide one class that matches structurally and demonstrate it.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a `.py` file defining a `TypedDict` `Book` with `title: str` and `pages: int`, a function using it, and one deliberately wrong literal (missing a key or wrong type). Run `mypy` on it and observe the error.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python -m mypy cell here


---

## Solutions

**Solution 1**

In [ ]:
def multiply(a: int, b: int) -> int:
    return a * b

print(multiply(6, 7))

**Solution 2**

In [ ]:
def names_over(people: list[str], threshold: int) -> list[str]:
    return [name for name in people if len(name) > threshold]

print(names_over(["Al", "Beatrice", "Sam", "Jonathan"], 3))

**Solution 3**

In [ ]:
def safe_divide(a: float, b: float) -> float | None:
    if b == 0:
        return None
    return a / b

result = safe_divide(10, 0)
if result is not None:
    print(result)
else:
    print("division by zero avoided")

**Solution 4**

In [ ]:
from typing import Protocol

class Closeable(Protocol):
    def close(self) -> None:
        ...

def shut(item: Closeable) -> None:
    item.close()

class File:
    def close(self) -> None:
        print("file closed")

shut(File())

**Solution 5**

In [ ]:
%%writefile sol5.py
from typing import TypedDict

class Book(TypedDict):
    title: str
    pages: int

def summary(b: Book) -> str:
    return f"{b['title']} has {b['pages']} pages"

ok: Book = {"title": "Dune", "pages": 412}
print(summary(ok))

bad: Book = {"title": "Pen"}      # missing 'pages' -> mypy error

In [ ]:
!python -m mypy sol5.py

---

## Summary and Key Points

- Type hints annotate variables, parameters, and returns; Python does not enforce them at runtime, but they document intent and enable static checking.
- Use built-in generics (`list[int]`, `dict[str, float]`) and the modern union syntax `X | Y` (Python 3.10+) instead of older `typing` equivalents.
- `Optional[T]` (or `T | None`) marks possibly-absent values, `Callable[[...], R]` types functions, `Any` opts out of checking, and type aliases name complex types.
- `Protocol` defines structural types (any object with the right shape matches, formalising duck typing); `TypedDict` describes dictionaries with specific keys and value types.
- `mypy` reads the annotations and reports inconsistencies with file and line, catching bugs before runtime; teams run it in continuous integration and adopt types gradually.

### Next module: 6.4 — Code Quality & Linting

The next module covers style and consistency: PEP 8, linters and auto-formatters such as `ruff`, `black`, and `isort`, and enforcing standards automatically with pre-commit hooks.